# Sample-Specific Variance in Differential Analysis

This tutorial demonstrates how to account for biological variability between samples when performing differential analysis in single-cell data. We'll explore:

1. Why sample variance matters and how it impacts statistical significance
2. How to run Kompot analyses with and without sample variance correction
3. How to visualize and interpret the results from both approaches
4. The biological implications of accounting for sample-specific effects

## Introduction to Sample Variance

When analyzing single-cell data across multiple biological samples (e.g., different patients, mice, or experimental batches), accounting for sample-specific variability is crucial. The biological variation between samples is often larger than the variation between conditions, which can lead to inflated statistical significance and false positive results if not properly addressed.

Kompot provides built-in support for sample variance estimation that:

- Builds separate estimators for each sample to capture sample-specific patterns
- Computes variance across samples within each condition group
- Adjusts statistical significance based on the observed sample-to-sample variability
- Produces more conservative and biologically meaningful differential results

## Setup and Data Loading

Let's load the necessary libraries and set up our environment:

In [ ]:
import anndata as ad
import matplotlib.pyplot as plt

# Import necessary libraries
import numpy as np
import palantir
import pandas as pd
import scanpy as sc
import seaborn as sns

import kompot

# Set plotting style
plt.rcParams["axes.spines.right"] = False
plt.rcParams["axes.spines.top"] = False
plt.rcParams["image.cmap"] = "Spectral_r"

In this tutorial, we'll analyze a bone marrow dataset containing cells from young and old mice. We'll compare these two age groups while properly accounting for biological variation across individual samples within each age group.

In [ ]:
# Data path - replace with your own AnnData file path
DATA_PATH = "../data/murine_bone_marrow_aging.h5ad"

# Analysis parameters
SAMPLE_COLUMN = "Replicate"  # Column in adata.obs with sample labels
GROUPING_COLUMN = "Age"  # Column in adata.obs with condition labels
CONDITIONS = ["Young", "Old"]  # Conditions to compare (first is reference)
CELL_TYPE_COLUMN = "highres_celltype"  # Column in adata.obs with cell type annotations
DIMENSIONALITY_REDUCTION = (
    "DM_EigenVectors"  # Key in adata.obsm for cell state representation
)
LAYER_FOR_EXPRESSION = (
    "logged_counts"  # Layer in adata.layers for expression data (None uses adata.X)
)

### Download dataset if not already present

This cell checks will download the paper dataset from https://zenodo.org/records/15587768 if not already presewnt.

In [ ]:
import os
import requests
from tqdm.auto import tqdm

# Primary download URL (Zenodo)
URL = "https://zenodo.org/records/15587768/files/murine_bone_marrow_aging.h5ad?download=1"

os.makedirs(os.path.dirname(DATA_PATH), exist_ok=True)

if not os.path.exists(DATA_PATH):
    print(f"Downloading dataset from: {URL}")
    response = requests.get(URL, stream=True)
    total = int(response.headers.get("content-length", 0))
    with open(DATA_PATH, "wb") as file, tqdm(
        desc=os.path.basename(DATA_PATH),
        total=total,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
            bar.update(len(chunk))
else:
    print(f"Dataset already available at {DATA_PATH}")

In [ ]:
adata = ad.read_h5ad(DATA_PATH)
adata

## Exploring the Data

Before performing differential analysis, it's important to understand the structure of your dataset. Let's visualize the data.

In [ ]:
sc.pl.umap(adata, color=CELL_TYPE_COLUMN)

In [ ]:
palantir.utils.run_diffusion_maps(adata, pca_key="X_pca_harmony", n_components=40);

## Standard Differential Analysis (Without Sample Variance)

First, we'll demonstrate the conventional differential analysis approach that doesn't account for sample-specific variability. This is the typical approach used when:
- There's only one sample per condition
- You're intentionally pooling all cells from each condition
- You want to establish a baseline for comparison with sample-variance-aware methods

### Step 1: Differential Abundance Analysis

We'll start by computing differential abundance between young and old conditions. This analyzes changes in cell state distribution between conditions:

In [ ]:
# First, let's compute differential abundance between conditions
da_results = kompot.compute_differential_abundance(
    adata,  # AnnData object
    groupby=GROUPING_COLUMN,  # Column with condition labels
    condition1=CONDITIONS[0],  # Reference condition
    condition2=CONDITIONS[1],  # Comparison condition
    obsm_key=DIMENSIONALITY_REDUCTION,  # Cell state representation
)

### Step 2: Differential Expression Analysis

Next, we'll compute differential expression between the conditions using the same standard approach (without sample variance). This identifies genes that change in expression between young and old conditions:

In [ ]:
# Now, compute differential expression between the conditions
de_results = kompot.compute_differential_expression(
    adata,  # AnnData object
    groupby=GROUPING_COLUMN,  # Column with condition labels
    condition1=CONDITIONS[0],  # Reference condition
    condition2=CONDITIONS[1],  # Comparison condition
    layer=LAYER_FOR_EXPRESSION,  # Expression data layer
    obsm_key=DIMENSIONALITY_REDUCTION,  # Cell state representation
    batch_size=0,  # set to, e.g., 100 to batch cells and genes for lower memory demand
)

## Differential Analysis With Sample Variance

Now, we'll perform the same analyses while properly accounting for sample-specific variability. The key difference is the addition of the `sample_col` parameter, which tells Kompot which column in `adata.obs` contains the sample identifiers.

### Step 1: Differential Abundance with Sample Variance

First, we'll compute differential abundance while accounting for sample variability:

In [ ]:
# First, let's compute differential abundance between conditions
da_results = kompot.compute_differential_abundance(
    adata,  # AnnData object
    groupby=GROUPING_COLUMN,  # Column with condition labels
    condition1=CONDITIONS[0],  # Reference condition
    condition2=CONDITIONS[1],  # Comparison condition
    obsm_key=DIMENSIONALITY_REDUCTION,  # Cell state representation
    sample_col=SAMPLE_COLUMN,  # Sample labels
)

### Step 2: Differential Expression with Sample Variance

When incorporating sample variance in differential expression analysis, memory usage can increase significantly because Kompot computes a gene-specific covariance matrix for each gene. To manage memory efficiently, we'll analyze only the top 200 genes (by Mahalanobis distance) from our previous analysis. We'll also enable disk storage for large matrices. Note that we are disableing the computation of a null distribution and therefore any false discovery rates with `null_genes=0`. Computing a null distribution with sample variance is very costly as separate covariance matrices have to be computed for each gene.

In [ ]:
# Now, compute differential expression between the conditions
topn = 200 # use a subset of genes to reduce resource demand
genes = adata.var.sort_values("kompot_de_Young_to_Old_mahalanobis", ascending=False).head(topn).index
de_results = kompot.compute_differential_expression(
    adata,  # AnnData object
    groupby=GROUPING_COLUMN,  # Column with condition labels
    condition1=CONDITIONS[0],  # Reference condition
    condition2=CONDITIONS[1],  # Comparison condition
    layer=LAYER_FOR_EXPRESSION,  # Expression data layer
    obsm_key=DIMENSIONALITY_REDUCTION,  # Cell state representation
    batch_size=0,  # set to, e.g., 100 to batch-process cells for lower memory demand
    store_arrays_on_disk=True, # stores covariance matrices on disk to process larger amounts of genes at once
    sample_col=SAMPLE_COLUMN,  # Sample labels
    genes=genes, # genes to be processed
    null_genes=0, # disable costly fdr computation
)

## Comparing Results: With vs. Without Sample Variance

Now let's compare the results from both approaches to understand how accounting for sample variance affects our differential analysis outcomes.

### Comparing Differential Abundance Results

First, we'll visualize the differential abundance results on the UMAP embedding. We'll compare:
1. The direction of abundance changes (increasing or decreasing)
2. The magnitude of log fold changes in abundance
3. The statistical significance (z-scores) with and without sample variance correction

In [ ]:
sc.pl.embedding(
    adata,
    "umap",
    color=[
        "kompot_da_log_fold_change_direction_Young_to_Old",
        "kompot_da_log_fold_change_Young_to_Old",
        "kompot_da_log_fold_change_zscore_Young_to_Old",
        "kompot_da_log_fold_change_zscore_Young_to_Old_sample_var",
    ],
    title=[
        "Abundacy Changes From Young to Old",
        "Log-Fold Changes From Young to Old",
        "Log-Fold Changes z-score without Sample Variance",
        "Log-Fold Changes z-score with Sample Variance",
    ],
    color_map="RdBu_r",
    vcenter=0,
    ncols=2,
)

### Volcano Plots for Differential Abundance

Volcano plots provide an effective way to visualize differential abundance results. They display the log fold change (effect size) on the x-axis and statistical significance on the y-axis. Points above the horizontal line and outside the vertical lines represent statistically significant changes.

Let's first examine the standard analysis without sample variance correction. To specify which data the plotting function should use, we can either provide the `lfc_key` and `ptp_key` columns with results created by the `compute_differential_abundance` function, or provide the `run_id` that identifies the differential abundance run (the first run has `run_id=0`):

In [ ]:
kompot.plot.volcano_da(adata, color=CELL_TYPE_COLUMN, run_id=0)

Now, let's examine the analysis with sample variance correction (`run_id=1`):

In [ ]:
kompot.plot.volcano_da(adata, color=CELL_TYPE_COLUMN, run_id=1)

### Key Observations from Differential Abundance Analysis

Comparing the volcano plots with and without sample variance correction reveals several important differences:

1. **Reduced statistical significance**: Posterior Tail Probabilities (y-axis) are generally lower when using sample variance correction, reflecting a more conservative statistical assessment that accounts for biological variability.

2. **Consistent cell type patterns**: Both plots show similar cell type-specific patterns of differential abundance, but with different significance thresholds.

3. **Unchanged effect sizes**: The log fold changes (x-axis) remain identical between methods, as sample variance affects statistical significance but not the estimation of effect size.

## Comparing Differential Expression Results

Now let's examine how sample variance correction affects differential gene expression analysis.

### Top Differentially Expressed Genes

First, let's examine the top differentially expressed genes identified in our standard analysis (without sample variance correction):

In [ ]:
adata.var[
    ["kompot_de_mean_lfc_Young_to_Old", "kompot_de_mahalanobis_Young_to_Old"]
].sort_values("kompot_de_mahalanobis_Young_to_Old", ascending=False).head(20)

### Volcano Plots for Differential Expression

Now, let's visualize differential expression results using volcano plots. First, we'll look at the standard analysis without sample variance correction (`run_id=0`):

In [ ]:
kompot.plot.volcano_de(
    adata,
    n_top_genes=100,
    run_id=0,
    title="Top 100 Gene Volcano Young to Old without Sample Variance",
)

Next, let's look at the analysis with sample variance correction (`run_id=1`). Notice how some genes show markedly lower significance (Mahalanobis distance) when accounting for sample-to-sample variability:

In [ ]:
kompot.plot.volcano_de(
    adata,
    n_top_genes=100,
    run_id=1,
    title="Top 100 Gene Volcano Young to Old with Sample Variance",
)

### Heatmaps of Top Differentially Expressed Genes

Heatmaps provide a cell type-specific view of expression changes. The expression values shown are averages of the actual expression values per cell type group, while the gene ranking is determined by the Mahalanobis distance computed by Kompot.

Let's first create a heatmap for the top genes identified without sample variance correction (`run_id=0`):

In [ ]:
kompot.plot.heatmap(
    adata,
    n_top_genes=20,
    groupby=CELL_TYPE_COLUMN,
    exclude_groups="Plasma cell",
    run_id=0,
    vmin="p1",
    vmax="p99",
)

Now, showing the top 20 genes when ranking them considering sample variance (`run_id=1`):

To understand why sample variance matters, let's examine genes that show high variability between samples. The following heatmap displays genes with the lowest significance scores after sample variance correction, comparing expression between replicate 1 and replicate 3:

In [ ]:
kompot.plot.heatmap(
    adata,
    genes=adata.var.sort_values(
        "kompot_de_mahalanobis_Young_to_Old_sample_var", ascending=False, na_position='first'
    ).tail(20).index,
    groupby=CELL_TYPE_COLUMN,
    condition_column=SAMPLE_COLUMN,
    condition1="1",
    condition1_name="Replicate 1",
    condition2="3",
    condition2_name="Replicate 3",
    vmin="p2",
    vmax="p98",
    cluster_rows=False,
)

### Detailed Expression Visualization

For specific genes of interest, we can create detailed visualizations that show:
1. Original expression in the dataset
2. Imputed expression in each condition (young and old)
3. Log fold change between conditions

Since these visualizations are not affected by sample variance correction, we can let the plotting function use the latest results from either analysis.

Let's look at a T-cell marker gene (Cd3g):

In [ ]:
kompot.plot.plot_gene_expression(adata, gene="Cd3g", vmin="p2", vmax="p98")

### Inspecting Run Parameters and Results Tracking

Kompot tracks all parameters and AnnData manipulations for each analysis run. This information can be inspected and compared between runs using the [RunInfo](https://kompot.readthedocs.io/en/latest/anndata.html#kompot.anndata.utils.RunInfo) utility and its `.compare_to()` method.

Let's examine the first differential expression run (`run_id=0`) without sample variance:

> **Note**: Some results have been overwritten by the sample variance-aware run (`run_id=1`). However, results that are not affected by sample variance (like log fold changes) remain identical between runs since all other parameters were the same. Results affected by sample variance are saved with a `_sample_var` suffix (e.g., `kompot_de_mahalanobis_Young_to_Old_sample_var`) to avoid naming conflicts.

In [ ]:
kompot.RunInfo(adata, run_id=0, analysis_type="de")

## Conclusion: The Impact of Sample Variance Correction

After comparing analyses with and without sample variance correction, several key insights emerge:

1. **Reduced false positives**: Sample variance correction significantly reduces the number of statistically significant findings, particularly for genes and cell states with high variability between samples.

2. **More robust biomarkers**: Genes that remain significant after sample variance correction are more likely to represent robust biological differences between conditions rather than sample-specific effects or technical artifacts.

3. **Preserved effect sizes**: The log fold changes themselves remain identical with both methods; only the statistical significance (Posterior Tail Probabilities/Mahalanobis distances) changes when accounting for sample variability.

4. **Cell type-specific reliability**: Some cell types show more consistent changes across samples than others, which becomes apparent when using sample variance correction.

### Recommendations

For experiments with multiple biological replicates per condition, we strongly recommend using sample variance correction to:

- Obtain more conservative and reliable significance estimates
- Reduce false positive findings due to sample-specific effects
- Identify the most robust biological signals that are consistent across samples
- Generate more reproducible results that are likely to validate in follow-up studies